# PRE_03_sql_transformation

## Objetivo
Solución docente ejecutable del minicaso.

## Ejecución
Ejecute las celdas de arriba abajo desde este directorio.

## Solución
La celda siguiente materializa los datos y artefactos definidos por el PRE.

In [ ]:
"""Construye un staging con versiones y su vista curada."""
import csv
import sqlite3
from pathlib import Path

ROOT = next(path for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (path / "data").is_dir() and (path / "submission").is_dir())
DATABASE, OUTPUT = ROOT / "data/customers_staging.db", ROOT / "submission/customers_curated.csv"
ROWS = [(1,"C001","Ana Ruiz","Bogota","Premium","42","2026-01-03"),(2,"C002","Luis Gomez","Medellin","premium","35","2026-01-04"),(3,"C003","Mariana Torres","Cali","CORPORATE","","2026-01-05"),(4,"C003","Mariana Torres","Cali","corporate","41","2026-02-10"),(5,"C004","Diego Lara","Bogota","","29","2026-01-09"),(6,"C005","Sara Paz","Cali","Premium","","2026-01-12"),(7,"C006","Juan Sol","Medellin","premium","38","2026-01-20"),(8,"C006","Juan Sol","Medellin","Premium","39","2026-02-01"),(9,"C007","Lina Rey","Bogota","CORPORATE","44","2026-01-18"),(10,"C008","Omar Gil","Cali","premium","31","2026-01-22")]


def build_submission():
    DATABASE.unlink(missing_ok=True)
    with sqlite3.connect(DATABASE) as db:
        db.execute("CREATE TABLE customer_staging (record_id INTEGER PRIMARY KEY, customer_id TEXT, customer_name TEXT, city TEXT, segment TEXT, age TEXT, updated_at TEXT)")
        db.executemany("INSERT INTO customer_staging VALUES (?, ?, ?, ?, ?, ?, ?)", ROWS)
        result = db.execute("""WITH ranked AS (
SELECT customer_id, customer_name, city,
CASE LOWER(segment) WHEN 'premium' THEN 'Premium' WHEN 'corporate' THEN 'Corporate' ELSE 'Unknown' END AS segment,
CAST(NULLIF(age, '') AS INTEGER) AS age, updated_at,
ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY updated_at DESC, record_id DESC) AS version_rank
FROM customer_staging)
SELECT customer_id, customer_name, city, segment, age, updated_at FROM ranked WHERE version_rank = 1 ORDER BY customer_id""").fetchall()
    with OUTPUT.open("w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file); writer.writerow(["customer_id","customer_name","city","segment","age","updated_at"]); writer.writerows(result)


if __name__ == "__main__":
    build_submission()


## Verificación
Revise los artefactos generados en `submission/` y las pruebas automatizadas del PRE.